# PREFIRE Virtual Datacube Generation - and storage using `Icechunk`

## Overview

Create an Icechunk store for a virtual datacube of PREFIRE satellite data that is ready for performing analyses without downloading files.

## Quick Start

**To run this notebook:**
1. Update configuration parameters in Section 1.1
2. Run all cells in order
3. Find your Icechunk store at the specified output location

## Prerequisites
- **NASA Earthdata account** ([sign up here](https://urs.earthdata.nasa.gov/users/new))
- **earthaccess**: NASA data search and authentication
- **virtualizarr**: Virtual dataset creation
- **xarray**: Multidimensional data handling
- **obstore**: Cloud storage access
- **icechunk**: Version-controlled virtual zarr store

## Virtual vs Traditional Approach

* **Traditional:** Download files (GB-TB) → Load into memory (or run out of memory) → Analyze 
* **Virtual:** Create lightweight reference files (MB) → Access on-demand (without downloading) → Analyze at scale

---

## 1. Setup and Authentication

In [ ]:
import warnings
from pathlib import Path
from urllib.parse import urlparse

import earthaccess
import xarray as xr
import virtualizarr as vz
from tqdm.notebook import tqdm

# VirtualiZarr components
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry

# Object store components  
from obstore.store import S3Store
from obstore.auth.earthdata import NasaEarthdataCredentialProvider

In [ ]:
# For Icechunk stuff
import tempfile
from datetime import datetime

import icechunk as ic
from icechunk import IcechunkStore, S3StaticCredentials, s3_storage

### Configuration

In [ ]:
# ============ CONFIGURATION ============
# Modify these settings to customize the workflow

# Dataset parameters
DATASET_SHORT_NAME = "PREFIRE_SAT2_3-SFC-SORTED-ALLSKY"
DATASET_VERSION = "R01"
# TEMPORAL_RANGE = ("2025-06-30 12:00", "2025-07-01 12:00")  # Uncomment to limit

# Processing parameters - to create the datacube
GROUP_NAMES = ["", "Sfc-Sorted"]
CONCATENATION_DIMENSION = "time"
XARRAY_CONCAT_OPTIONS = {
    "coords": "minimal",
    "compat": "override",
    "combine_attrs": "override",
}

# Output configuration
USE_LOCAL_STORAGE = True  # Set False for S3 storage
LOCAL_TEMP_DIR = None  # None = auto-generate temp directory
OUTPUT_PREFIX = "prefire_icechunk"

# Credentials
S3_REGION = "us-west-2"
credential_endpoint_mapping = {
    "asdc-prod-protected": "https://data.asdc.earthdata.nasa.gov/s3credentials",
}

#### Earthdata Login and data provider credentials

In [ ]:
try:
    auth = earthaccess.login()
    print(f"✓ Authenticated as {auth.username}" if auth.authenticated else "❌ Authentication failed")
except Exception as e:
    print(f"❌ Login failed: {e}")
    print("Check credentials, or visit: https://urs.earthdata.nasa.gov/users/new for account setup")
    raise

In [ ]:
# We can use the NASA credential provider to automatically refresh credentials
try:
    cp = NasaEarthdataCredentialProvider(credential_endpoint_mapping["asdc-prod-protected"])
    print("✓ Credential provider initialized")
except Exception as e:
    print(f"❌ Failed to initialize credential provider: {e}")
    raise

## 2. Build Virtual Datacube

Create a virtual representation of PREFIRE data without downloading files:

1. **Search** cloud-hosted NetCDF files
2. **Create** virtual datasets for each file and group
3. **Combine** into unified structure

**PREFIRE Data Structure**

- *Root group* (`""`): Global metadata and coordinates
- *Sfc-Sorted group*: Surface-sorted emission data

We'll process both groups separately, then merge them.

### 2.1. Search NASA Data Archive

Search for PREFIRE satellite files in NASA's cloud storage.

In [ ]:
# Retrieve data files for the dataset I'm interested in
print(f"Searching for PREFIRE Level 3 surface-sorted data...")
search_params = {
    "short_name": DATASET_SHORT_NAME,
    "version": DATASET_VERSION,
    "cloud_hosted": True,
}

# Add temporal filter if configured
# if 'TEMPORAL_RANGE' in globals():
#     search_params["temporal"] = TEMPORAL_RANGE

results = earthaccess.search_data(**search_params)

if not results:
    raise ValueError("No PREFIRE data found - check search parameters")
print(f"✓ Found {len(results)} granules")

In [ ]:
# Get S3 endpoints for all files:
s3_data_links = [g.data_links(access="direct")[0] for g in results]
parsed_s3_url = urlparse(s3_data_links[0])

print("First few granules: \n  {}".format('\n  '.join(s3_data_links[0:3])))
print(f"\nparsed = {parsed_s3_url}")
print(f"Bucket: {parsed_s3_url.netloc}")

s3_store = S3Store(
    bucket=parsed_s3_url.netloc,
    region="us-west-2",
    credential_provider=cp,
    virtual_hosted_style_request=False,
    client_options={"allow_http": True},
)
object_registry = ObjectStoreRegistry({f"s3://{parsed_s3_url.netloc}": s3_store})

s3_store

In [ ]:
# # Un-comment this cell if you want to download the original files.
# files_downloaded = earthaccess.download(results[0])
# dtree = xr.open_datatree(files_downloaded[0])
# dtree

### 2.2. Set up virtual dataset creation function

In [ ]:
def create_virtual_dataset(file_path: str, group_name: str):
    """Create virtual dataset from NetCDF group without loading data.

    Without downloading files, build lightweight references to cloud data, containing:
        - **Metadata**: Dimensions, coordinates, attributes
        - **Chunk references**: Pointers to specific byte ranges in cloud storage
        - **No actual data**: Just instructions on how to fetch it when needed
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Numcodecs codecs*", category=UserWarning)
        return open_virtual_dataset(
            file_path, 
            parser=HDFParser(group=group_name),
            registry=object_registry
        )

### 2.3. Test a single file

In [ ]:
print("Testing virtual dataset creation...")
test_vds = create_virtual_dataset(file_path=s3_data_links[0], group_name="Sfc-Sorted")
print(f"✓ Test successful. Dataset shape: {test_vds.sizes}")

### 2.4. Process All Files

In [ ]:
%%time
print(f"Processing {len(s3_data_links)} files across {len(GROUP_NAMES)} groups...")

vdatasets_by_group = {group: [] for group in GROUP_NAMES}

for file_index, file_path in enumerate(tqdm(s3_data_links, desc="Processing files")):
    for group_name in tqdm(GROUP_NAMES, desc=f"Groups (file {file_index+1})", leave=False):
        virtual_ds = create_virtual_dataset(file_path, group_name)
        vdatasets_by_group[group_name].append(virtual_ds)
        
print(f"✓ Created {sum(len(d) for d in vdatasets_by_group.values())} virtual datasets")

In [ ]:
# Show what a virtual dataset looks like
print(f"\nSample virtual dataset structure:")
sample_vds = vdatasets_by_group["Sfc-Sorted"][0]
sample_vds

In [ ]:
%%time
# Concatenate datasets for each group.
concatenated_by_group = {}
for group_name, group_datasets in vdatasets_by_group.items():
    concatenated_by_group[group_name] = xr.concat(
        group_datasets, dim=CONCATENATION_DIMENSION, **XARRAY_CONCAT_OPTIONS
    )
    print(f"  ✓ Group '{group_name}': {concatenated_by_group[group_name].sizes}")

In [ ]:
%%time
# Merge the groups into one dataset.
datacube = xr.merge([vds for _, vds in concatenated_by_group.items()])

print(f"✓ Consolidated datacube created with {len(datacube.data_vars)} variables")
print(f"  Time dimension: {datacube.sizes.get('time', 'N/A')} steps")

datacube

## 3. Export to Icechunk

Following: https://icechunk.io/en/latest/quickstart/#installation

### 3.1. Helper functions

Source: adapted from Julius Busecke's [icechunk_opener.py](https://github.com/jbusecke/earthaccess/blob/icechunk_opener/earthaccess/icechunk_opener.py)

These functions handle authentication with NASA Earthdata for accessing
virtual chunk data through Icechunk repositories.

In [ ]:
def _get_credential_endpoint(url: str) -> str:
    sep = "/"
    parsed = urlparse(url)
    if parsed.scheme != "s3":
        raise ValueError(
            "Only s3 is supported as storage protocol. Got {parsed.protocol}"
        )
        # TODO: Is protocol the right vocabulary here?
    bucket_w_prefix_full = parsed.netloc + parsed.path.rstrip(sep)
    components = bucket_w_prefix_full.split(sep)

    while len(components) > 0:
        partial_target = sep.join(components)
        if partial_target in credential_endpoint_mapping.keys():
            return credential_endpoint_mapping[partial_target]
        components = components[0:-1]

    raise ValueError("Could not find any matching credential endpoint for {url}")

In [ ]:
class S3IcechunkCredentials:
    def __init__(self, endpoint: str):
        self.endpoint = endpoint

    def __call__(self) -> S3StaticCredentials:
        if not earthaccess.__auth__.authenticated:
            raise ValueError(
                "A valid Earthdata login instance is required to retrieve credentials for icechunk stores"
            )
        creds = earthaccess.__auth__.get_s3_credentials(endpoint=self.endpoint)
        if len(creds) == 0:
            raise ValueError(f"Got no credentials from endpoint {self.endpoint}")
        return S3StaticCredentials(
            access_key_id=creds["accessKeyId"],
            secret_access_key=creds["secretAccessKey"],
            expires_after=datetime.fromisoformat(creds["expiration"]),
            session_token=creds["sessionToken"],
        )

In [ ]:
def get_virtual_chunk_credentials(
    storage: ic.Storage,
) -> dict[str, ic.AnyCredential | None]:
    """Function to retrieve virtual chunk containers from icechunk storage and authenticate
    all allowed virtual chunk prefixes using EDL credentials.
    """
    # get config and extract virtual containers
    config = ic.Repository.fetch_config(storage=storage)
    # TODO: accommodate case without virtual chunk containers.
    if not config:
        msg = f"Got empty config from {storage=} and will not try to infer chunk containers. If this is a native store this behavior is expected if the config was not persisted to storage."
        warnings.warn(message=msg)
        vchunk_container_urls: List[str] = []
    else:
        vchunk_containers = config.virtual_chunk_containers
        if vchunk_containers:
            vchunk_container_urls = list(vchunk_containers.keys())
        else:
            raise ValueError("No virtual chunk containers found.")

    # try to build authentication for all virtual chunk containers. If any of the virtual
    # chunk containers is not 'approved' it will raise an error in `_get_credential_endpoint`.
    # We will catch the error here, warn, and only return the authenticated urls.
    # Users will then get an error for the remaining containers and need to add those manually!
    failed_container_urls = []
    credential_mapping = {}
    for url in vchunk_container_urls:
        try:
            endpoint = _get_credential_endpoint(url)
            credential_mapping[url] = ic.s3_refreshable_credentials(
                S3IcechunkCredentials(endpoint=endpoint)
            )
        except ValueError:
            failed_container_urls.append(url)

    if len(failed_container_urls) > 0:
        # TODO: link to credentials in icechunk + docs about the endpoint registry
        warnings.warn(
            f"Could not build virtual chunk credentials for {failed_container_urls}.\
                      If the URL is a non EDL bucket, you have to manually construct credentials (...)"
        )

    # TODO: Check how easy it is to 'splice' this output with manually created credentials
    return ic.containers_credentials(credential_mapping)

### 3.2. Configure Storage Location

In [ ]:
# Parse source data URL for virtual chunk container
parsed = urlparse(s3_data_links[0])
bucket = parsed.netloc
prefix = parsed.path.lstrip("/")
bucket_path = f"s3://{bucket}/"

print(f"Virtual chunk container (bucket): {bucket_path}")

# Get credentials for accessing virtual chunks
endpoint = _get_credential_endpoint(s3_data_links[0])
cred_provider = S3IcechunkCredentials(endpoint=endpoint)
static_creds = cred_provider()

virtual_credentials = ic.containers_credentials({
    bucket_path: ic.s3_credentials(
        access_key_id=static_creds.access_key_id, 
        secret_access_key=static_creds.secret_access_key
    )
})

print(f"✓ Credentials configured for {bucket_path}")

In [ ]:
# Initialize Icechunk configuration
config = ic.RepositoryConfig.default()

config.set_virtual_chunk_container(
    ic.VirtualChunkContainer(
        bucket_path,
        store=ic.s3_store(region="us-west-2"),
    )
)

In [ ]:
# Define the S3 Virtual Chunk Container
storage = s3_storage(
    bucket=bucket,
    prefix=prefix,
    region="us-west-2",
    get_credentials=cred_provider,
)

### 3.3. Create Repository Storage

In [ ]:
if USE_LOCAL_STORAGE:
    # Local storage (for testing/development)
    if LOCAL_TEMP_DIR:
        storage_path = Path(LOCAL_TEMP_DIR)
        storage_path.mkdir(parents=True, exist_ok=True)
    else:
        tempdir = tempfile.TemporaryDirectory()
        storage_path = Path(tempdir.name)
    
    repo_storage = ic.local_filesystem_storage(str(storage_path))
    print(f"✓ Using local storage: {storage_path}")
else:
    # S3 storage (for production/sharing)
    # TODO: Add S3 storage configuration
    raise NotImplementedError("S3 storage not yet configured")

Note: Temporary directory will be cleaned up when Python exits

For persistent storage, set `LOCAL_TEMP_DIR` to a permanent location

In [ ]:
# Create/Open repository
repo = ic.Repository.open_or_create(
    storage=repo_storage,
    config=config,
    authorize_virtual_chunk_access=virtual_credentials,
)

### 3.4. Write (commit) the virtual dataset into icechunk

In [ ]:
try:
    # Create a writable session on the default main branch.
    session = repo.writable_session("main")
    # Send the virtual references to the Icechunk store, and commit them.
    datacube.vz.to_icechunk(session.store)

    commit_msg = f"Initial datacube creation - {DATASET_SHORT_NAME} - {datetime.now().strftime('%Y-%m-%d')}"
    commit_id = session.commit(commit_msg)
    print(f"✓ Datacube committed successfully")
    print(f"  Commit ID: {commit_id}")
    
except Exception as e:
    print(f"❌ Error writing to Icechunk: {e}")
    raise

## 4. Load and Visualize Datacube

This section demonstrates how to:
1. Re-open an existing Icechunk repository
2. Access the virtual datacube
3. Create visualizations using cloud data on-demand

In [ ]:
import zarr

In [ ]:
# Open authenticated icechunk repo
virtual_chunk_credentials = get_virtual_chunk_credentials(repo_storage)

repo = ic.Repository.open(
    storage=repo_storage, 
    authorize_virtual_chunk_access=virtual_chunk_credentials
)

In [ ]:
# View all snapshots/commits on the main branch
for ancestor in repo.ancestry(branch="main"):
    print(ancestor)

In [ ]:
# We can also list all branches in the repository.
print(repo.list_branches())

In [ ]:
# Open a read-only session on the 'main' branch
session = repo.readonly_session("main")

# Access the Zarr store from this session
store = session.store

In [ ]:
# Open the root group
root = zarr.open_group(store=store, mode="r")

# List all arrays and groups in the store
print(list(root.array_keys()))   # Shows array names
print(list(root.group_keys()))   # Shows sub-group names
print(root.tree())               # Shows a visual hierarchy (if using zarr-python)

In [ ]:
ds = xr.open_zarr(store)
ds

In [ ]:
data_variable = ds["emis_mean"]
data_variable

### 4.2. Create visualization

We will create a visualization to demonstrate virtual datacube functionality similar to xarray usage with a standard dataset.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from xarray.plot.utils import label_from_attrs

In [ ]:
# projection = ccrs.PlateCarree()
projection = ccrs.Orthographic(0, 90)

def make_nice_map(axis):
    axis.add_feature(cfeature.OCEAN, color='lightblue')
    axis.add_feature(cfeature.COASTLINE, edgecolor='grey')
    axis.add_feature(cfeature.BORDERS, edgecolor="grey", linewidth=0.5)

    grid = axis.gridlines(draw_labels=["left", "bottom"], dms=True, linestyle=':')
    grid.xformatter = LONGITUDE_FORMATTER
    grid.yformatter = LATITUDE_FORMATTER

In [ ]:
print("Creating sample visualization from virtual datacube...")
print("Note: Data is read on-demand from cloud storage during plotting")

# Extract data for plotting
time_idx, xtrack_idx, surface_idx, spectral_idx = 0, 5, 7, 5

x = ds["longitude"][time_idx, :, :].squeeze()
y = ds["latitude"][time_idx, :, :].squeeze()
values = data_variable[time_idx, xtrack_idx, surface_idx, :, :, spectral_idx].squeeze()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": projection})

make_nice_map(ax)

contour_handle = ax.contourf(
    x, y, values,
    transform=ccrs.PlateCarree(),
    alpha=0.8,
    zorder=2,
)

cb = plt.colorbar(contour_handle)
cb.set_label(label_from_attrs(values))

plt.show()